In [ ]:
#### Minnesota Vikings Bye Week Performance Analysis #####
# This is an analysis of Post Bye Week performance of the Minnesota Vikings
# The goal is to evaluate whether or not the first game after a bye week is an advantage or not
#
# Hypothesis: Since the Bye Week provides the team with an extra week of rest and preperation for their next game, the team should perform lead to more wins than loses
# Definition: To be considered an "advantage", the team should win at least 66% of games after a bye week, which is significantly higher than the 50% win rate expected by chance.
# How is a bye week defined? A BYE week is for this exercise is defined as any time the team has 13 or more days of rest in between games
#                            NOTE: We do not consider just 14 days as a bye week, as there are occasions when games are played on Thursday, Saturdays and Mondays that can throw off the 14 rest period.

In [5]:
# Import the necessary libraries
import nfl_data_py as nfl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from pathlib import Path

print("All libraries imported successfully!")
print("Pandas version:", pd.__version__)

# Cache a copy of the schedule data locally to avoid repeated downloads
path = Path('data') / 'full_schedule.csv'
if os.path.exists(path):
    print("Local copy of schedule data found. Loading from file...")
    schedules = pd.read_csv(path)
else:
    schedules = nfl.import_schedules(list(range(1999, 2025)))
    os.makedirs('.\\data', exist_ok=True)
    schedules.to_csv(path, index=False)
    print("Schedules downloaded and saved locally")

All libraries imported successfully!
Pandas version: 3.0.2
Local copy of schedule data found. Loading from file...


In [ ]:
# Filter out the Vikings games from the rest of the schedule

vikingsSchedule = schedules[(schedules['home_team'] == 'MIN') | (schedules['away_team'] == 'MIN')]
print("Vikings schedule data loaded successfully!")

# Print and verify the schedule
print(vikingsSchedule.head())

In [ ]:
# Not every column is needed. We will filter down to the columns that we need

neededCols = [
    'game_id',     # database ID of the game
    'season',      # Season(year) of the game
    'game_type',   # Regular season, pre season, or play offs
    'week',        # Week of the season (1-18 for regular season, 1-4 for pre season, 1-5 for play offs)
    'home_team',   # Home team abbreviation
    'away_team',   # Away team abbreviation
    'home_score',  # Home team score
    'away_score',  # Away team score
    'result',      # Result of the game (home win, away win, tie) If the result is POSITIVE, the home team won. If the result is NEGATIVE, the away team won. If the result is 0, the game was a tie.
    'home_rest',   # Number of days of rest for the home team before the game
    'away_rest',   # Number of days of rest for the away team before the game
    'div_game',    # Whether the game is a divisional game or not
    'location'     # Location of the game (home, away, neutral)
]

filteredVikingsSchedule = vikingsSchedule[neededCols]
print("Filtered schedule data to only include needed columns")

# Create a new column to indicate if the Vikings won the game or not
filteredVikingsSchedule['vikings_win'] = ((filteredVikingsSchedule['home_team'] == 'MIN') & (filteredVikingsSchedule['result'] > 0)) | ((filteredVikingsSchedule['away_team'] == 'MIN') & (filteredVikingsSchedule['result'] < 0))

# Create a new column to indicate if the game was after a bye week or not
filteredVikingsSchedule['had_bye_week'] = (filteredVikingsSchedule['home_rest'] >= 13) | (filteredVikingsSchedule['away_rest'] >= 13)